# CS5100 — Assignment 5 (FA25)
**Name:** Xinle Xu  
**NUID:** 002335847


In [ ]:
# imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

import scipy.cluster.hierarchy as shc
from sklearn.datasets import load_iris
from sklearn.cluster import AgglomerativeClustering, KMeans

np.random.seed(42)
Path("figures").mkdir(exist_ok=True)
plt.rcParams["figure.dpi"] = 120

def plot_clusters(X, labels, title, centers=None, ax=None):
    ax = ax or plt.gca()
    ax.scatter(X[:, 0], X[:, 1], c=labels, s=28)
    if centers is not None:
        ax.scatter(centers[:, 0], centers[:, 1], marker="X", s=120, edgecolor="k")
    ax.set_xlabel("Sepal length (cm)")
    ax.set_ylabel("Sepal width (cm)")
    ax.set_title(title)


In [ ]:
# data: sepal length/width only
iris = load_iris()
cols = ["sepal length (cm)", "sepal width (cm)"]
df = pd.DataFrame(iris.data, columns=iris.feature_names)[cols]
X = df.values
data = X  # alias if needed

plt.figure(figsize=(4.8, 3.6))
plt.scatter(X[:, 0], X[:, 1], s=20)
plt.xlabel("Sepal length (cm)")
plt.ylabel("Sepal width (cm)")
plt.title("Iris scatter (sepal-only)")
plt.tight_layout()
plt.savefig("figures/scatter_sepal.png", bbox_inches="tight")
plt.show()


In [ ]:
# dendrogram (ward)
plt.figure(figsize=(10, 7))
plt.title("Iris Dendrograms")
dend = shc.dendrogram(shc.linkage(X, method='ward'), no_labels=True)
plt.xticks([])
plt.tight_layout()
plt.savefig("figures/dendrogram_ward.png", bbox_inches="tight")
plt.show()


In [ ]:
# dendrogram with cut lines
Z = shc.linkage(X, method='ward')
plt.figure(figsize=(9.2, 3.6))
shc.dendrogram(Z, no_labels=True)
for k in [2, 3, 4, 5]:
    t = Z[-(k-1), 2] - 1e-6
    plt.axhline(t, ls="--", lw=1)
plt.title("Dendrogram (ward) with cut lines: k = 2, 3, 4, 5")
plt.tight_layout()
plt.savefig("figures/dendrogram_ward_cuts.png", bbox_inches="tight")
plt.show()


In [ ]:
# agglomerative: ward vs complete for k in {2,3,4,5}
candidate_k = [2, 3, 4, 5]
for k in candidate_k:
    fig, axes = plt.subplots(1, 2, figsize=(9.6, 3.6), sharex=True, sharey=True)
    for method, ax in zip(["ward", "complete"], axes):
        agg = AgglomerativeClustering(
            n_clusters=k,
            linkage=method,
            metric="euclidean"
        )
        labels = agg.fit_predict(X)
        plot_clusters(X, labels, f"Agglomerative ({method}, k={k})", ax=ax)
    plt.tight_layout()
    plt.savefig(f"figures/agglomerative_{k}.png", bbox_inches="tight")
    plt.show()


In [ ]:
# k-means with the same k; show centers
for k in [2, 3, 4, 5]:
    km = KMeans(n_clusters=k, n_init=10, random_state=42)
    labels = km.fit_predict(X)
    centers = km.cluster_centers_
    plt.figure(figsize=(4.8, 3.6))
    plot_clusters(X, labels, f"K-Means (k={k})", centers=centers)
    plt.tight_layout()
    plt.savefig(f"figures/kmeans_{k}.png", bbox_inches="tight")
    plt.show()


## Answers

**Q1.** With only sepal length/width, one cluster stands clearly apart; the other two blend along an oblique band. Picking three groups is defensible, but the latter boundary is inherently fuzzy in this 2D view.

**Q2.** Ward tends to form compact, variance-minded groups; Complete uses farthest-pair distance, giving tighter borders and sometimes peeling off small fringes. In the plots, Ward looks more balanced; Complete can isolate tails and make stricter cuts.

**Q3.** K-Means assumes centroided, roughly convex blobs and pulls edge points toward the nearest mean. Agglomerative has no centroids; the linkage defines how sets merge and exposes structure across scales. Here, K-Means yields crisp centers; Agglomerative changes the cut with linkage (Ward vs Complete), slicing the mixed region a bit differently.


In [ ]:
# versions (optional)
import sys, sklearn, scipy
print("Python:", sys.version.split()[0])
print("numpy:", np.__version__)
print("pandas:", pd.__version__)
print("matplotlib:", plt.matplotlib.__version__)
print("scikit-learn:", sklearn.__version__)
print("scipy:", scipy.__version__)
